In [ ]:
import pandas as pd
import glob
import os

def verileri_birlestir_ve_temizle(klasor_yolu=".", cikis_dosyasi="tum_is_ilanlari_final.csv"):
    dosyalar = glob.glob(os.path.join(klasor_yolu, "*.csv"))
    print(f"{len(dosyalar)} adet CSV dosyası bulundu.")

    df_listesi = []

    for dosya in dosyalar:
        if os.path.basename(dosya) == cikis_dosyasi:
            continue
            
        try:
            df = pd.read_csv(dosya, encoding='utf-8-sig')
            df_listesi.append(df)
            print(f"Okundu: {dosya} - Satır sayısı: {len(df)}")
        except Exception as e:
            print(f"Hata oluştu ({dosya}): {e}")

    birlestirilmis_df = pd.concat(df_listesi, axis=0, ignore_index=True, sort=False)
    
    ilk_sayi = len(birlestirilmis_df)
    
    birlestirilmis_df = birlestirilmis_df.drop_duplicates().reset_index(drop=True)
    
    son_sayi = len(birlestirilmis_df)
    
    birlestirilmis_df.to_csv(cikis_dosyasi, index=False, encoding='utf-8-sig')
    
    print("\n--- İŞLEM TAMAMLANDI ---")
    print(f"Birleşen toplam satır: {ilk_sayi}")
    print(f"Silinen tekrar eden satır: {ilk_sayi - son_sayi}")
    print(f"Final satır sayısı: {son_sayi}")
    print(f"Dosya kaydedildi: {cikis_dosyasi}")

verileri_birlestir_ve_temizle()

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_excel("tum_is_ilanlari_final.xlsx")

def veriyi_duzelt_garanti(df):
    df.columns = [str(col).strip() for col in df.columns]
    
    gruplar = {
        'Baslik': ['Pozisyon', 'Unvan', 'title', 'pozisyon', 'Aranan_Pozisyon', 'İlan_Başlığı', 'Başlık', 'job_title', 'baslik'],
        'Sirket': ['Sirket', 'company', 'sirket', 'Şirket', 'company_name', 'company'],
        'Lokasyon': ['Lokasyon', 'remote', 'location', 'konum', 'Şehir', 'lokasyon'],
        'Metin': ['Aciklama', 'Ilan_Metni', 'description', 'ilan_metin', 'job_description', 'Tam_Metin', 'Nitelikler', 'Kriterler']
    }

    for yeni_ad, eski_adlar in gruplar.items():
        mevcutlar = [c for c in eski_adlar if c in df.columns]
        if mevcutlar:
            df[yeni_ad] = df[mevcutlar].bfill(axis=1).iloc[:, 0]
        else:
            df[yeni_ad] = "Belirtilmemiş"

    yetenekler = ['Python', 'SQL', 'Excel', 'İngilizce', 'İletişim', 'Liderlik', 'Analiz', 'Takım_Calısması', 'Agile']
    
    for y in yetenekler:
        if y in df.columns:
            df[y] = pd.to_numeric(df[y], errors='coerce').fillna(0).astype(int)
        else:
            df[y] = 0

    tutulacak_sutunlar = ['Baslik', 'Sirket', 'Lokasyon', 'Metin'] + yetenekler
    mevcut_tutulacaklar = [c for c in tutulacak_sutunlar if c in df.columns]
    
    df_final = df[mevcut_tutulacaklar].copy()

    duplike_kontrol = [c for c in ['Baslik', 'Sirket'] if c in df_final.columns]
    if duplike_kontrol:
        df_final = df_final.drop_duplicates(subset=duplike_kontrol).reset_index(drop=True)

    df_final['Beceri_Sayisi'] = df_final[yetenekler].sum(axis=1)

    return df_final

df_temiz = veriyi_duzelt_garanti(df)

df_temiz.to_excel("analize_hazir_is_ilanlari.xlsx", index=False)
print("Düzeltme tamamlandı!")
print(f"Yeni Sütunlar: {df_temiz.columns.tolist()}")
print(f"Toplam İlan Sayısı: {len(df_temiz)}")